<a id="top"></a>

# Lab L0709: validate tool calls (the application validates)

```yaml
title: "Lab L0709: validate tool calls (the application validates)"
keywords: validation, hallucinated tool call, tool_calls, dispatch, missing argument, invented tool, protocol, lab
estimated duration: 25
```

> **Lesson:** L07. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L07/objectives.md). Solutions: `L0709_lab_solutions.ipynb`. **Pure Python — no API key needed** (you validate crafted tool calls, the same offline approach as the L0708 demo's fallback cell).
>
> **After this lab you can:** write the validation step that catches a hallucinated tool call, and explain why the schema alone does not prevent one.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Write the validator](#2-problem-1--write-the-validator)
- [3. Problem 2 — Run it over every crafted call](#3-problem-2--run-it-over-every-crafted-call)
- [4. Problem 3 — The three outcomes (written)](#4-problem-3--the-three-outcomes-written)
- [5. Problem 4 — Why doesn't the schema stop a bad call? (written)](#5-problem-4--why-doesnt-the-schema-stop-a-bad-call-written)
- [6. Self-check](#6-self-check)

## 1. Setup

The `calculator` (which raises on non-arithmetic input), plus crafted tool calls — one good, three hallucinated — in the LangChain `{name, args, id}` shape an `AIMessage.tool_calls` entry carries.

In [ ]:
from typing import Annotated, Any


def calculator(
    expression: Annotated[str, "The arithmetic expression to evaluate, e.g. '18374 * 92431'."],
) -> str:
    """Evaluate a basic arithmetic expression (the four operators and parentheses) and
    return the exact numeric result. Use this whenever the user asks for a calculation.

    Raises ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


# Crafted tool calls a model might emit -- the LangChain {name, args, id} shape that
# an AIMessage.tool_calls entry carries (no live model needed to practice validation).
CALLS: list[dict[str, Any]] = [
    {"name": "calculator", "args": {"expression": "47219 * 8803"}, "id": "c1", "type": "tool_call"},
    {"name": "calculator", "args": {"expression": "twenty + 1"}, "id": "c2", "type": "tool_call"},
    {"name": "calculator", "args": {}, "id": "c3", "type": "tool_call"},
    {"name": "wikipedia", "args": {"query": "geese"}, "id": "c4", "type": "tool_call"},
]
print(len(CALLS), "crafted calls ready")

[↑ Back to top](#top)

## 2. Problem 1 — Write the validator

Implement `validate_call(call)` for a tool call dict (read `call["name"]` and `call["args"]`) returning an `"OK: ..."` string with the result for a good call, or a `"REJECTED: ..."` string for each failure: an unknown tool name, a missing `expression` argument, or a `calculator` `ValueError`. The application — not the model — is responsible for catching all three.

In [ ]:
def validate_call(call: dict[str, Any]) -> str:
    """Validate one tool call (a {name, args, id} dict) and run it, or return a rejection string."""
    if call["name"] != "calculator":
        return f"REJECTED: no such tool {call['name']!r} (the model invented it)"
    args = call["args"]
    if "expression" not in args:
        return f"REJECTED: missing required argument 'expression' (got {args!r})"
    try:
        return f"OK: calculator({args['expression']!r}) = {calculator(args['expression'])}"
    except ValueError as exc:
        return f"REJECTED: {exc}"


print(validate_call(CALLS[0]))  # expect OK with a number

[↑ Back to top](#top)

## 3. Problem 2 — Run it over every crafted call

Loop over `CALLS` and print the verdict for each. Three of the four should be REJECTED.

In [ ]:
for call in CALLS:
    print(validate_call(call))

[↑ Back to top](#top)

## 4. Problem 3 — The three outcomes (written)

Name the **three** observable outcomes of handing a model a single tool, in one line each.

_Write your answer by editing this cell (double-click)._

1. **Called** — the model returns a well-formed tool call (a `calculator` call with a valid `expression`), and the application runs it.
2. **Answered** — the model replies with text and an empty `.tool_calls`; it judged the tool unnecessary.
3. **Malformed / hallucinated** — the model returns a tool call the application must reject: a missing or ill-typed argument, an invented argument, or a tool name that doesn't exist.

[↑ Back to top](#top)

## 5. Problem 4 — Why doesn't the schema stop a bad call? (written)

"The application validates; the model proposes." In a sentence or two: why does passing a JSON-Schema tool definition *not* prevent the model from emitting malformed or invented arguments?

_Write your answer by editing this cell (double-click)._

Because the schema is only a *description* handed to the model in the prompt, not a validator applied to its output. The model generates tokens shaped like a tool call from training; nothing at generation time forces those tokens to match the schema, so it can omit a required field, pass the wrong type, or invent a tool name. Enforcement happens only when *your* code checks the call before running it — which is why the application, not the schema, is what actually validates.

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0709_lab_solutions.ipynb`. Done when `validate_call` returns `OK` for the first call and `REJECTED` (with a useful reason) for the non-numeric, missing-arg, and invented-tool calls.

[↑ Back to top](#top)